# Feature — lm-eval Integration

In [ ]:
"""
REAL lm-eval standardized benchmarking via the documented LMEvalRunner.
=======================================================================

`agenttune.eval` exports `LMEvalConfig` / `LMEvalTask` / `LMEvalRunner` as the
integration with EleutherAI's `lm-eval` harness for standardized benchmarks::

    from agenttune.eval import LMEvalConfig, LMEvalTask, LMEvalRunner
    runner = LMEvalRunner(LMEvalConfig(model_name="..."))
    results = runner.evaluate_tasks([LMEvalTask(..., lm_eval_task_name="arc_challenge")])

`LMEvalRunner` shells out to the real `lm_eval` CLI (`--model hf ...`), then parses the
timestamped results JSON and extracts the metrics. This notebook runs two real standardized
benchmarks — **ARC-Challenge** and **WinoGrande** — on a live `Qwen/Qwen3-0.6B`, from the local
Hugging Face cache (no network), through the documented `LMEvalRunner.evaluate_tasks` path, and
reads back the real accuracies plus the combined summary the runner writes.

HONEST SCOPE — read before quoting
----------------------------------
This proves the *integration runs for real*: the real `lm_eval` subprocess
executes, produces a results file, and `LMEvalRunner` parses genuine metrics out
of it. To keep it fast it caps each task at a small `--limit`, so the accuracies
are **subset estimates on a 0.6B model, not leaderboard numbers** — do not quote
them as ARC/WinoGrande scores. The claim is that the documented runner works, not
that this model is good.

Needs `Qwen/Qwen3-0.6B` and the `allenai/ai2_arc` (ARC-Challenge) + `winogrande`
datasets in the local Hugging Face cache.
"""
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")        # subprocess inherits → uses cache only
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

In [2]:
import glob
import json

In [3]:
from agenttune.eval import LMEvalConfig, LMEvalTask, LMEvalRunner

In [4]:
MODEL = "Qwen/Qwen3-0.6B"
OUT_DIR = "/tmp/agenttune-lm-eval"
LIMIT = 30  # samples per task — keeps it fast; makes these SUBSET estimates, not full scores

In [5]:
TASKS = [
    LMEvalTask(
        name="arc_challenge",
        category="commonsense_reasoning",
        description="AI2 ARC-Challenge multiple-choice science questions.",
        lm_eval_task_name="arc_challenge",
        metrics=["acc", "acc_norm"],
    ),
    LMEvalTask(
        name="winogrande",
        category="commonsense_reasoning",
        description="WinoGrande pronoun-resolution commonsense task.",
        lm_eval_task_name="winogrande",
        metrics=["acc"],
    ),
]

In [6]:
def main():
    config = LMEvalConfig(
        model_name=MODEL,
        batch_size=16,
        limit=LIMIT,
        output_dir=OUT_DIR,
        save_results=True,
    )
    runner = LMEvalRunner(config)

    print(f"\n── Running real lm_eval ({MODEL}, limit={LIMIT}/task, from HF cache) ──")
    print("   (each task shells out to the real `lm_eval --model hf` CLI)")
    results = runner.evaluate_tasks(TASKS)

    print("\n── Parsed metrics (read back from the real lm_eval results JSON) ──")
    ok_tasks = 0
    for r in results:
        err = "error" in r.metrics
        acc = r.metrics.get("acc")
        print(f"  {r.task_name:<14} metrics={ {k: round(v, 4) for k, v in r.metrics.items()} }")
        if not err and isinstance(acc, (int, float)) and 0.0 <= acc <= 1.0:
            ok_tasks += 1

    # The runner writes a combined summary across tasks — confirm it is real.
    summary_file = os.path.join(OUT_DIR, "lm_eval_summary.json")
    summary = {}
    if os.path.exists(summary_file):
        with open(summary_file) as f:
            summary = json.load(f)

    # Confirm the real lm_eval results files exist on disk (proof the CLI ran).
    result_jsons = [p for p in glob.glob(os.path.join(OUT_DIR, "*.json"))
                    if "summary" not in os.path.basename(p)
                    and "samples" not in os.path.basename(p)]

    print("\n── Summary ──")
    print(f"  tasks with a real accuracy in [0,1]   : {ok_tasks}/{len(TASKS)}")
    print(f"  lm_eval results files on disk         : {len(result_jsons)}")
    print(f"  combined summary written by runner    : {bool(summary)}")
    if summary.get("summary"):
        print(f"  summary stats (subset, NOT leaderboard): "
              f"{ {k: round(v, 4) for k, v in summary['summary'].items()} }")
    print("  scope: integration runs for real; small-subset estimates, not full-benchmark scores")

    assert ok_tasks == len(TASKS), "not every task returned a real accuracy in [0,1] (lm_eval/parse failed)"
    assert len(result_jsons) >= len(TASKS), "lm_eval results JSON missing — the CLI did not produce output"
    assert summary and summary.get("summary"), "combined summary not written by the runner"
    print("\n✓ LMEvalRunner ran the real lm_eval harness end-to-end and parsed real scores.")

In [7]:
if __name__ == "__main__":
    main()


── Running real lm_eval (Qwen/Qwen3-0.6B, limit=30/task, from HF cache) ──
   (each task shells out to the real `lm_eval --model hf` CLI)


2026-08-04:08:41:38 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.


2026-08-04:08:41:51 INFO     [_cli.run:388] Selected Tasks: ['arc_challenge']


2026-08-04:08:41:53 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-08-04:08:41:53 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen3-0.6B'}


2026-08-04:08:41:58 INFO     [models.huggingface:286] Using device 'cuda:0'


...

Running loglikelihood requests: 100%|██████████| 60/60 [00:00<00:00, 82.92it/s]


fatal: not a git repository (or any parent up to mount point /workspace)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


2026-08-04:08:42:50 INFO     [loggers.evaluation_tracker:247] Saving results aggregated
2026-08-04:08:42:50 INFO     [loggers.evaluation_tracker:119] Saving per-task samples to /tmp/agenttune-lm-eval/*.jsonl


hf ({'pretrained': 'Qwen/Qwen3-0.6B'}), gen_kwargs: ({}), limit: 30.0, num_fewshot: None, batch_size: 16
|  Tasks   |Version|Filter|n-shot|Metric|   |Value|   |Stderr|
|----------|------:|------|-----:|------|---|----:|---|-----:|
|winogrande|      1|none  |     0|acc   |↑  |  0.6|±  | 0.091|




── Parsed metrics (read back from the real lm_eval results JSON) ──
  arc_challenge  metrics={'acc': 0.3333, 'acc_norm': 0.4}
  winogrande     metrics={'acc': 0.6}

── Summary ──
  tasks with a real accuracy in [0,1]   : 2/2
  lm_eval results files on disk         : 2
  combined summary written by runner    : True
  summary stats (subset, NOT leaderboard): {'acc_mean': 0.4667, 'acc_max': 0.6, 'acc_min': 0.3333, 'acc_norm_mean': 0.4, 'acc_norm_max': 0.4, 'acc_norm_min': 0.4}
  scope: integration runs for real; small-subset estimates, not full-benchmark scores

✓ LMEvalRunner ran the real lm_eval harness end-to-end and parsed real scores.
